# Assignment 2 — Evaluating the Impact of FOMC Communications on Asset Prices

**FRE-GY 7871 A · NLP and the Investment Process · Fall 2026**

Kevin Warsh became Fed Chair on 22 May 2026. This notebook (1) collects FOMC statements, minutes and the Chair's speeches/testimony from Feb 2018 (Powell baseline) to today, (2) scores each document hawkish/dovish with **two methods** (a monetary-policy word list, and FinBERT sentiment / factor similarity), (3) measures the one-day reaction of four indicators and regresses them on tone controlling for the 3-month bill yield, and (4) forecasts the September 2026 FOMC meeting.

Run the cells top to bottom. Output is saved so the notebook stands on its own.

In [ ]:
import sys
from pathlib import Path
import warnings

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make `import src.*` work from the repo root regardless of the kernel CWD.
ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT))

from src import fomc
from src.config import WARSH_START, POWELL_START
from src.tone_wordlist import score_dataframe as score_wordlist
from src.regress import one_day_change, run_regressions

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
print("OK")

## Step 1 — Collect the documents

Three kinds of FOMC communication, all from federalreserve.gov:

- **press releases** = post-meeting FOMC statements
- **meeting minutes** (full text, followed from the announcement page)
- **the Chair's speeches**, including congressional **testimony** and **press-conference transcripts**

Each document keeps its release **date and time**. The Chair label switches from Powell to Warsh at 22 May 2026.

In [ ]:
docs = fomc.collect()
stmt_dates = docs.loc[docs.kind == "statement", "date"]
presconf = fomc.collect_press_conferences(stmt_dates)
docs = fomc.merge_docs(docs, presconf)
fomc.save_texts(docs)

print(f"{len(docs)} documents total")
docs.head()

### Table 1 — documents collected, by type and by Chair

In [ ]:
table1 = pd.crosstab(docs["kind"], docs["chair"], margins=True)
table1

> **Write-up note.** The Warsh era spans only ~4 months (22 May – 13 Sep 2026), so the Warsh column is tiny relative to Powell's ~8 years. State this small-sample caveat explicitly; it is central to how much the forecast can rely on the Warsh-era regressions.

## Step 2 — Tone scoring (hawkish / dovish)

Two methods, as required.

**Method 1 — word list.** A monetary-policy lexicon of hawkish phrases (`"higher inflation"`, `"inflation pressures"`, `"rate hike"`, …) and dovish phrases (`"inflation has eased"`, `"rate cut"`, `"softening"`, …). Net score `wl_net_share = (n_hawk − n_dove)/(n_hawk + n_dove)` ∈ [−1, +1] (positive = hawkish). *Refine the lexicon in `src/tone_wordlist.py` against the actual documents.*

**Method 2 — FinBERT.** Either sentence-level sentiment (positive/negative probability) or cosine similarity to key sentences (`"Interest rates will rise"`, `"Inflation will rise"` vs `"Interest rates will fall"`, `"Inflation will ease"`), per "Parsing the Fed". Requires `transformers` + `torch`.

In [ ]:
scored = score_wordlist(docs)

try:
    from src.tone_finbert import score_dataframe as score_finbert
    scored = score_finbert(scored)
except ImportError as e:
    print("FinBERT not available (transformers/torch missing); word list only:", e)

scored.to_csv("data/scores/tone_scores.csv", index=False)
scored[["kind", "date", "chair", "wl_net_share", "wl_n_hawk", "wl_n_dove"]].head()

### Figure 1 — hawkish/dovish tone over time by document type

Mark the start of Warsh's term (22 May 2026) with a vertical line. Change `tone_col` to swap the word-list score for a FinBERT column (e.g. `fb_sim_net`).

In [ ]:
tone_col = "wl_net_share"   # or "fb_sim_net", "fb_pos", ...

fig, ax = plt.subplots(figsize=(11, 5.5))
for kind, grp in scored.groupby("kind"):
    ax.plot(grp["date"], grp[tone_col], marker="o", ls="-", ms=4, label=kind)
ax.axvline(pd.Timestamp(WARSH_START), color="r", ls="--", lw=1.5, label="Warsh term start")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("release date")
ax.set_ylabel(f"{tone_col} (positive = hawkish)")
ax.set_title("Hawkish/dovish tone over time by document type")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

## Step 3 — Market reaction and validation

For each release we compute the **one-day change** in the four indicators (`post_close − prev_close`), then regress each on each tone score, controlling for the **3-month Treasury bill** change (`DGS3MO`) so the words are not credited with the rate decision itself.

In [ ]:
from src.market import download_fred, download_yahoo

fred = download_fred()
yahoo = download_yahoo()

tone_cols = [c for c in scored.columns
             if c.startswith(("wl_", "fb_")) and c != "wl_n_words"]

event = one_day_change(fred, yahoo, scored.set_index("doc_id")["date"])
event = event.join(scored.set_index("doc_id")[["kind", "chair"] + tone_cols])
event.to_csv("data/scores/event_changes.csv", index=False)
event[["release", "dxy_chg", "t10y2y_chg", "dgs1_chg", "gv_chg"]].tail()

### Table 2 — one-day change in the four indicators after each Warsh-era release

In [ ]:
warsh_events = event[event.chair == "Warsh"].sort_values("release")
cols = ["release", "kind", tone_col] + ["dxy_chg", "t10y2y_chg", "dgs1_chg", "gv_chg"]
warsh_events[cols].round(4)

### Table 3 — each indicator's one-day change regressed on each tone score (3M-bill control)

In [ ]:
regs = run_regressions(event, tone_cols)
regs.to_csv("data/scores/regressions.csv", index=False)
regs.round(4)

## Step 4 — Forecast of the September 2026 FOMC meeting

Combine the trend (Step 2) and the validated market reaction (Step 3) into explicit probabilities. Fill in the numbers below — they must be *your* judgment, grounded in the tables above.

In [ ]:
# Rate decision probabilities -- MUST sum to 100%.
p_cut, p_hold, p_hike = 0.00, 0.00, 0.00
assert abs((p_cut + p_hold + p_hike) - 1.0) < 1e-9, "probabilities must sum to 1"

# Statement tone: probability the September statement is more hawkish than July's.
p_more_hawkish = 0.00

# Market reaction: for each indicator, P(rises on statement day) and expected size.
market_forecast = {
    "DXY":         {"p_rise": 0.00, "size": 0.00},
    "10s2s":       {"p_rise": 0.00, "size": 0.00},
    "1Y yield":    {"p_rise": 0.00, "size": 0.00},
    "Growth-Value":{"p_rise": 0.00, "size": 0.00},
}

pd.DataFrame([
    ["cut / hold / hike", f"{p_cut:.0%} / {p_hold:.0%} / {p_hike:.0%}"],
    ["statement more hawkish", f"{p_more_hawkish:.0%}"],
] + [[k, f"{v['p_rise']:.0%} (size {v['size']:+.3f})"]
     for k, v in market_forecast.items()],
    columns=["item", "forecast"])

## Recommendation

> **Write-up (your own words):** one position you would take ahead of the meeting, why your analysis (Steps 2–3) supports it, and the specific outcome that would prove you wrong.

## Comparison with the readings

> **Write-up:** how your methods and results compare with Doh–Kim–Yang (2021), Doh–Song–Yang (2020), and "Parsing the Fed". Note in particular that the alternative-statements method is unavailable for the recent sample (5-year release lag), which is why this notebook uses the word list + FinBERT instead.